# LLM Analysis Figures

This notebook is intended to reproduce the LLM plots in the paper. It assumes that LLM representations and layerwise metrics have already been generated by the corresponding scripts (see details below).

Prerequisite metric are read from `results/llms/metrics/<method>/`, and figures are saved to `results/llms/figs/`.

## Prerequisite: LLM Reps Extraction and Metrics Computation

Example commands for one model:

```bash
uv run python scripts/llm_analysis/extract_llm_representations.py   --model-name meta-llama/Llama-3.1-8B   --model-key llama   --dataset wikitext   --shard aa   --data-file data/raw/texts/wikitext/shard_aa   --quantization 8bit

uv run python scripts/llm_analysis/compute_layerwise_metrics.py --model llama --dataset wikitext --shard aa --method gride
uv run python scripts/llm_analysis/compute_layerwise_metrics.py --model llama --dataset wikitext --shard aa --method entropy
uv run python scripts/llm_analysis/compute_layerwise_metrics.py --model llama --dataset wikitext --shard aa --method avg_l2
uv run python scripts/llm_analysis/compute_layerwise_metrics.py --model llama --dataset wikitext --shard aa --method avg_cosine
```

Run the analogous metric commands for `mistral` and `pythia` before executing all plotting cells.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from rethinking_neural_id.paths import RepoPaths
from rethinking_neural_id.plotting.llm import (
    DEFAULT_LLM_MODELS,
    MODEL_COLORS,
    layer_slice,
    llm_figs_dir,
    load_avg_cosine,
    load_avg_l2,
    load_entropy,
    load_gride,
)

paths = RepoPaths.default()
fig_dir = llm_figs_dir(paths)

models = list(DEFAULT_LLM_MODELS)
dataset = "wikitext"
shard = "aa"

print(f"Loading LLM metrics from: {paths.llm_metrics_root}")
print(f"Saving figures to: {fig_dir}")


def save_figure(fig, filename: str):
    output_path = fig_dir / filename
    fig.savefig(output_path, bbox_inches="tight")
    print(f"Saved: {output_path}")
    return output_path

## ID and Entropy Across Layers

In [ ]:
fs = 14
line_w_id = 2.8
line_w_ent = 2.2
alpha_line = 0.7
sl = layer_slice(exclude_first=True)

fig, ax_id = plt.subplots(figsize=(8, 4))
ax_ent = ax_id.twinx()
handles_models = []
last_x = None

for model in models:
    color = MODEL_COLORS.get(model, ax_id._get_lines.get_next_color())
    id_avg = load_gride(model, dataset=dataset, shard=shard, paths=paths, sl=sl)["id"]
    entropy = load_entropy(model, dataset=dataset, shard=shard, paths=paths, sl=sl)

    L = int(min(len(id_avg), len(entropy)))
    if L == 0:
        continue
    id_avg = id_avg[:L]
    entropy = entropy[:L]
    x = np.arange(1, L + 1)
    last_x = x

    id_line, = ax_id.plot(
        x, id_avg, color=color, linewidth=line_w_id, alpha=alpha_line, label=model
    )
    ax_ent.plot(
        x, entropy, color=color, linewidth=line_w_ent, alpha=alpha_line, linestyle="--"
    )
    handles_models.append(id_line)

ax_id.set_xlabel("Layer", fontsize=fs)
ax_id.set_ylabel("ID estimate (solid)", fontsize=fs)
ax_ent.set_ylabel("Entropy (dashed)", fontsize=fs)
ax_id.tick_params(labelsize=fs - 1)
ax_ent.tick_params(labelsize=fs - 1)
ax_id.grid(axis="y", alpha=0.25)
ax_id.grid(axis="x", alpha=0.25)
if last_x is not None:
    ax_id.set_xlim(0, last_x[-1] + 0.5)

fig.subplots_adjust(right=0.76)
ax_id.legend(
    handles_models,
    [handle.get_label() for handle in handles_models],
    title="Model",
    fontsize=fs - 1,
    title_fontsize=fs - 1,
    loc="center left",
    bbox_to_anchor=(1.16, 0.55),
)
fig.tight_layout()

save_figure(fig, f"id_vs_entropy_layers_{'_'.join(models)}.pdf")
plt.show()

## GRIDE ID Estimates Across Layers

In [ ]:
line_w_avg = 4
line_w_scaling = 2
alpha_scaling = 0.28
alpha_ci = 0.15
fs = 15
sl = layer_slice(exclude_first=True)

fig, ax = plt.subplots(figsize=(8, 5))
last_x = None

for model in models:
    gride = load_gride(model, dataset=dataset, shard=shard, paths=paths, sl=sl)
    id_ls = gride["id_ls"]
    id_avg = gride["id"]
    err_ls = gride["err_ls"]

    L, S = id_ls.shape
    x = np.arange(1, L + 1)
    last_x = x
    color = MODEL_COLORS.get(model, ax._get_lines.get_next_color())

    for s in range(S):
        ax.plot(
            x,
            id_ls[:, s],
            linewidth=line_w_scaling,
            alpha=alpha_scaling,
            color=color,
            linestyle=":" if s == 0 else "-",
            zorder=1,
        )

    if err_ls is not None and np.all(np.isfinite(err_ls)):
        mean_err = np.nanmean(err_ls, axis=1)
        ax.fill_between(
            x, id_avg - 2.0 * mean_err, id_avg + 2.0 * mean_err,
            color=color, alpha=alpha_ci, linewidth=0, zorder=2,
        )

    ax.plot(
        x,
        id_avg,
        linewidth=line_w_avg,
        alpha=0.98,
        color=color,
        label=model,
        zorder=3,
    )

ax.set_xlabel("Layer", fontsize=fs)
ax.set_ylabel("Estimated ID", fontsize=fs)
ax.tick_params(labelsize=fs - 1)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", alpha=0.25)
if last_x is not None:
    ax.set_xlim(0, last_x[-1] + 1)
ax.legend(
    title="Model",
    fontsize=fs - 1,
    title_fontsize=fs - 1,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
)
fig.tight_layout()

save_figure(fig, f"gride_ids_layers_{'_'.join(models)}.pdf")
plt.show()

## GRIDE NN-Distances Across Layers

In [ ]:
line_w_avg = 2
line_w_scaling = 2
alpha_scaling = 0.4
fs = 15
sl = layer_slice(exclude_first=True, exclude_last=True)

fig, ax = plt.subplots(figsize=(8, 4))
last_x = None

for model in models:
    gride = load_gride(model, dataset=dataset, shard=shard, paths=paths, sl=sl)
    r_ls = gride["r_ls"]
    r_avg = gride["r"]

    L, S = r_ls.shape
    x = np.arange(1, L + 1)
    last_x = x
    color = MODEL_COLORS.get(model, ax._get_lines.get_next_color())

    for s in range(S):
        ax.plot(
            x,
            r_ls[:, s],
            linewidth=line_w_scaling,
            alpha=alpha_scaling,
            color=color,
            linestyle=":" if s == 0 else "-",
            zorder=1,
        )

    ax.plot(
        x,
        r_avg,
        linewidth=line_w_avg,
        alpha=0.98,
        color=color,
        label=model,
        zorder=3,
    )

ax.set_xlabel("Layer", fontsize=fs)
ax.set_ylabel("NN-Distances", fontsize=fs)
ax.tick_params(labelsize=fs - 1)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", alpha=0.25)
if last_x is not None:
    ax.set_xlim(0, last_x[-1] + 1)
ax.legend(
    title="Model",
    fontsize=fs - 1,
    title_fontsize=fs - 1,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
)
fig.tight_layout()

save_figure(fig, f"gride_r_layers_{'_'.join(models)}.pdf")
plt.show()

## NN-Distances and Average Representation Length

In [ ]:
sl = layer_slice(exclude_first=True)
line_w_avg = 2
line_w_scaling = 2
alpha_scaling = 0.4
alpha_ci = 0.15
fs = 15

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(14, 4))
max_L_left = 0
max_L_right = 0

for model in models:
    gride = load_gride(model, dataset=dataset, shard=shard, paths=paths, sl=sl)
    r_ls = gride["r_ls"]
    r_avg = gride["r"]

    L, S = r_ls.shape
    x = np.arange(1, L + 1)
    color = MODEL_COLORS.get(model, ax_left._get_lines.get_next_color())

    for s in range(S):
        ax_left.plot(
            x,
            r_ls[:, s],
            linewidth=line_w_scaling,
            alpha=alpha_scaling,
            color=color,
            linestyle=":" if s == 0 else "-",
            zorder=1,
        )

    ax_left.plot(x, r_avg, linewidth=line_w_avg, alpha=0.98, color=color, label=model, zorder=3)
    max_L_left = max(max_L_left, L)

for model in models:
    mean, std = load_avg_l2(model, dataset=dataset, shard=shard, paths=paths, sl=sl)
    L = mean.shape[0]
    x = np.arange(1, L + 1)
    color = MODEL_COLORS.get(model, ax_right._get_lines.get_next_color())

    ax_right.plot(x, mean, linewidth=line_w_avg, alpha=0.98, color=color, label=model, zorder=3)
    ax_right.fill_between(
        x, mean - 2.0 * std, mean + 2.0 * std,
        color=color, alpha=alpha_ci, linewidth=0, zorder=2,
    )
    max_L_right = max(max_L_right, L)

ax_left.set_xlabel("Layer", fontsize=fs)
ax_left.set_ylabel("NN-Distances", fontsize=fs)
ax_left.tick_params(labelsize=fs - 1)
ax_left.grid(axis="y", alpha=0.25)
ax_left.grid(axis="x", alpha=0.25)
ax_left.set_xlim(0, max_L_left + 1)

ax_right.set_xlabel("Layer", fontsize=fs)
ax_right.set_ylabel("Avg. representation length", fontsize=fs)
ax_right.tick_params(labelsize=fs - 1)
ax_right.grid(axis="y", alpha=0.25)
ax_right.grid(axis="x", alpha=0.25)
ax_right.set_xlim(0, max_L_right + 1)

legend_handles = [
    Line2D([0], [0], linestyle="-", linewidth=line_w_avg, markersize=6,
           color=MODEL_COLORS[model], label=model)
    for model in models
]
plt.subplots_adjust(right=0.82, wspace=0.28)
fig.legend(
    handles=legend_handles,
    labels=models,
    fontsize=fs - 1,
    title_fontsize=fs - 1,
    loc="center left",
    bbox_to_anchor=(0.86, 0.5),
    frameon=False,
)
fig.tight_layout(rect=[0, 0, 0.865, 1], w_pad=2.5)

save_figure(fig, f"nn_and_l2_layers_{'_'.join(models)}.pdf")
plt.show()

## Average Cosine Similarity Across Layers

In [ ]:
sl = layer_slice(exclude_first=True)
fs = 18
lw = 2
alpha_line = 0.9
alpha_ci = 0.15
ms = 8

fig, ax = plt.subplots(figsize=(12.5, 6.0))
max_L = 0

for model in reversed(models):
    mean, std = load_avg_cosine(model, dataset=dataset, shard=shard, paths=paths, sl=sl)
    L = mean.shape[0]
    x = np.arange(1, L + 1)
    color = MODEL_COLORS[model]

    ax.plot(
        x,
        mean,
        "-o",
        linewidth=lw,
        markersize=ms,
        alpha=alpha_line,
        color=color,
        label=model,
    )
    ax.fill_between(
        x, mean - 2.0 * std, mean + 2.0 * std,
        color=color, alpha=alpha_ci, linewidth=0,
    )
    max_L = max(max_L, L)

ax.grid(axis="both", alpha=0.3)
ax.set_xlim(0, max_L + 1)
ax.set_xticks(np.arange(0, max_L + 1, max(1, max_L // 6)))
ax.tick_params(labelsize=fs - 2)
ax.set_xlabel("Layer", fontsize=fs)
ax.set_ylabel("Average Cosine Similarity", fontsize=fs)

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="-", linewidth=lw, markersize=ms,
           color=MODEL_COLORS[model], label=model)
    for model in models
]
plt.subplots_adjust(right=0.82)
fig.legend(
    handles=legend_handles,
    labels=models,
    fontsize=fs - 2,
    loc="center left",
    bbox_to_anchor=(0.845, 0.5),
    frameon=False,
)
fig.tight_layout(rect=[0, 0, 0.85, 1])

save_figure(fig, f"avg_cossim_{'_'.join(models)}.pdf")
plt.show()